# Step 15.4 — Implement `MiniGPT`：把 Top-Level Architecture 变成一个干净的 MLX `nn.Module`

Step 15.3 已经确定顶层结构：

```text
token IDs
↓
token embedding + learned position embedding
↓
Block × N
↓
final RMSNorm
↓
LM head
↓
logits
```

现在把它真正封装成：

```python
class MiniGPT(nn.Module):
    ...
```

本节重点不是增加新的模型数学，而是把已经学过的组件正确组织进 MLX module system，并守住三个实现边界：

1. **shape contract**：input 必须是 `(B,T)`，并且 `1 <= T <= T_max`；
2. **independent Blocks**：`Block × N` 是 N 个独立参数实例；
3. **structural constants are not trainable parameters**：causal mask / position IDs 这类运行时结构量不能误挂成普通 trainable array attribute。

本轮只实现 MiniGPT，不进入 Step 15.5 的完整 forward shape trace、总参数计算与逐行历史映射。


## 1. MLX `nn.Module` 到底会追踪什么？

MLX 0.32.2 的 `nn.Module` 会递归追踪 module state。

特别重要的是，赋给 module attribute 的这些对象会被纳入 module tree：

```text
mx.array
dict
list
tuple
```

其中 `parameters()` 会递归返回 module tree 中的 `mx.array` members；`trainable_parameters()` 则排除被冻结的 arrays。

所以：

```python
self.blocks = [
    TransformerBlock(...),
    TransformerBlock(...),
]
```

是合法的。MLX 会追踪 list 内的子 Modules；官方 MLX 模型示例也使用这种方式保存 layer stack。

相反：

```python
self.some_constant = mx.array(...)
```

如果没有 freeze，就会被当成 module array parameter/state，并默认进入 trainable parameter tree。

这正是 causal mask 需要特别小心的原因。


## 2. MiniGPT 中哪些东西应该 Trainable，哪些不应该？

### 应该 trainable

```text
token_embedding.weight
position_embedding.weight
每个 Block 内的 QKV / c_proj / Norm / MLP parameters
final_norm.weight
lm_head.weight
```

### 不应该 trainable

```text
当前 sequence length T
position IDs [0,1,...,T-1]
causal visibility mask
head_dim
1/sqrt(head_dim) scale constant
max_seq_len metadata
```

因此本节采用最简单、最安全的策略：

> **position IDs 和 causal mask 都在 forward 中按当前 T 临时构造，不作为 `self.xxx = mx.array(...)` 保存。**

`head_dim`、`scale`、`max_seq_len` 等保存为普通 Python int / float metadata，不是 learned arrays。


## 3. 为什么 Causal Mask 继续使用 Forward-Local Construction？

Attention 需要：

$$
M_T\in\mathbb R^{T\times T}.
$$

而当前 T 可以随着输入改变。

在 `__call__` 内创建：

```python
future = mx.triu(mx.ones((T,T)), k=1) > 0
mask = mx.where(future, -inf, 0)
```

有两个直接好处：

- mask 自动匹配当前 sequence length；
- 它只是 forward-local tensor，不会进入 `MiniGPT.trainable_parameters()`。

### 如果未来真的要缓存 Mask 呢？

MLX 也允许把 array 保存成 module state 后显式 freeze，例如概念上：

```python
self.causal_mask = cached_mask
self.freeze(
    recurse=False,
    keys='causal_mask',
    strict=True,
)
```

冻结后它仍属于 module state / parameters，但会被排除在 `trainable_parameters()` 之外。

本项目当前不需要缓存，所以优先使用 forward-local mask，保持实现更直接。


## 4. 先把已有的 Block 作为 MiniGPT 的内部 Building Block

本节沿用 Step 14.5 的设计，不改变模型约定：

```text
Causal Multi-Head Attention
  qkv_proj / c_proj: bias=False

Feed-Forward MLP
  C -> 4C -> GELU -> C
  bias=True

TransformerBlock
  Pre-Norm
  RMSNorm
```

Attention 内的 causal mask 继续保持 forward-local。

下面代码只是把已审核的已有组件整理成一个可被 `MiniGPT` 使用的自包含版本。


In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


## 5. `MiniGPT.__init__`：把顶层组件注册进 Module Tree

MiniGPT 构造函数需要：

```text
vocab_size
dims
num_heads
num_blocks
max_seq_len
```

并建立：

```text
token_embedding
position_embedding
blocks
final_norm
lm_head
```

其中：

```python
self.blocks = [
    TransformerBlock(...)
    for _ in range(num_blocks)
]
```

会创建 N 个**不同对象**，因此每个 Block 有自己独立的 learned parameters。

这和：

```python
block = TransformerBlock(...)
self.blocks = [block] * num_blocks
```

不同；后者会重复引用同一个对象，造成参数共享，不是本课程当前 MiniGPT 的设计。


## 6. `MiniGPT.__call__` 的 Forward Contract

输入 contract：

$$
token\_ids:(B,T)
$$

并要求：

$$
\boxed{1\le T\le T_{max}}.
$$

Token IDs 还应是合法 integer categorical indices：

$$
0\le token\_id<V.
$$

本节显式检查 tensor rank 与 context length；token dtype / index range 由调用者 contract 与 embedding lookup 共同约束，不在 forward 中做额外 device synchronization 式检查。

Forward：

$$
(B,T)
\xrightarrow{tok\_emb}
(B,T,C)
$$

$$
(T)
\xrightarrow{pos\_emb}
(T,C)
$$

相加后依次通过所有 Blocks、final norm、LM head，最终得到：

$$
\boxed{logits:(B,T,V)}.
$$


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


## 7. 为什么这些 Shape Checks 值得放进教学实现？

MiniGPT 顶层最容易出现的错误不是高深数学，而是 tensor contract 被破坏。

本实现检查：

```text
token_ids rank == 2
T >= 1
T <= max_seq_len
logits shape == (B,T,V)
```

而 `dims % num_heads == 0` 同时在 MiniGPT / Attention construction 阶段检查。

这些 checks 不是为了替代正式测试，而是把 architecture invariant 写进代码接口。

特别是：

$$
T>T_{max}
$$

时 position embedding table 已经没有对应 learned row，因此应该明确拒绝，而不是静默继续。


## 8. `self.blocks` 为什么要用“每次重新构造”的 List Comprehension？

正确：

```python
self.blocks = [
    TransformerBlock(...)
    for _ in range(N)
]
```

会得到：

```text
Block 0 -> parameter set 0
Block 1 -> parameter set 1
...
Block N-1 -> parameter set N-1
```

而如果先创建一个对象：

```python
shared = TransformerBlock(...)
self.blocks = [shared] * N
```

只是把同一个 Python object reference 放进去 N 次。

这会产生 weight sharing。

Weight sharing 并非数学上非法，但它不是当前 roadmap 中 `Block × N` 的默认 MiniGPT architecture，所以本实现明确避免这种写法。


## 9. 参数树里应该出现什么？

概念上，`model.trainable_parameters()` 应来自这些 learned arrays：

```text
token_embedding.weight
position_embedding.weight

blocks[0] ... blocks[N-1]
  attention projection weights
  RMSNorm weights
  MLP weights / biases

final_norm.weight
lm_head.weight
```

而不应该因为我们的实现方式额外出现：

```text
causal_mask
position_ids
sequence_length
scale constant
```

当前代码通过：

- mask 在 Attention `__call__` 内局部构造；
- position IDs 在 MiniGPT `__call__` 内局部构造；
- `scale` / `max_seq_len` 等使用 Python scalar metadata；

来保证这些 structural values 不会被意外纳入 trainable array tree。

本节不计算整个模型参数总数；那属于 Step 15.5。


## 10. Forward-Local Constant 与 Frozen State 的区别

当前 MiniGPT 的 causal mask：

```text
只在 forward 中存在
调用结束后不作为 model attribute 保存
不会出现在 model.parameters()
```

如果未来为了性能缓存：

```python
self.causal_mask = ...
```

那么 MLX 会把这个 `mx.array` 看成 module state / parameter array。

这时至少应该：

```python
self.freeze(
    recurse=False,
    keys='causal_mask',
    strict=True,
)
```

使其不进入 `trainable_parameters()`。

所以：

```text
forward-local structural tensor
≠
frozen module-state tensor
≠
trainable parameter
```

三者需要明确区分。


## 11. 一个最小 Usage Example（准备代码，不运行）

例如：

```text
V = 32
C = 16
H = 4
N = 2
T_max = 16
```

输入：

$$
token\_ids:(1,5)
$$

按照实现 contract，理论 output shape 应为：

$$
\boxed{(1,5,32)}.
$$

下面 cell 只是给未来 work copy 使用；本轮不执行，也不声称 assertion 已通过。


In [ ]:
# USAGE / SHAPE-CHECK CELL — prepared only, NOT executed here.

model = MiniGPT(
    vocab_size=32,
    dims=16,
    num_heads=4,
    num_blocks=2,
    max_seq_len=16,
)

sample_ids = mx.array([[1, 7, 3, 9, 2]])  # (B,T) = (1,5)
logits = model(sample_ids)

assert logits.shape == (1, 5, 32)
print('logits shape:', logits.shape)

# No forward call or assertion has been run in the master notebook.


## 12. 一个实现选择：本节暂时不做 Weight Tying

当前：

```text
token_embedding.weight : (V,C)
lm_head.weight          : (V,C)   # MLX stored layout
```

虽然 shape 相同，但本节把它们作为**两个独立 learned parameter arrays**。

有些 language models 会让 input token embedding 与 output LM-head weights 共享 / tie；这是可选 architecture design。

当前 roadmap 没有要求 weight tying，所以 MiniGPT 先保持独立，避免把另一个设计选择混进 15.4。

因此不要因为两个 weight shapes 都是 `(V,C)` 就自动认为它们是同一个参数。


## 13. Step 15.4 材料边界

- 本节实现完整 `MiniGPT(nn.Module)`：token embedding、learned absolute position embedding、独立 Block list、final RMSNorm、LM head。
- Input contract 为 integer token IDs `(B,T)`，并显式要求 `1<=T<=T_max`；constructor 显式检查 positive dimensions 与 `dims % num_heads == 0`。
- `self.blocks=[TransformerBlock(...) for ...]` 创建独立 Block objects；MLX 0.32.2 会递归追踪 list 中的子 Modules。
- causal mask 与 position IDs 都在 forward 内按当前 T 局部构造，不作为 `mx.array` module attributes，因此不会意外进入 trainable parameter tree。
- MLX Module 中直接保存的 `mx.array` 默认会被 parameters/trainable-parameters 机制追踪；若未来缓存 structural array，应显式 freeze 或采用其他非训练状态设计。
- 输出 contract 为 logits `(B,T,V)`；本节不加 softmax、不计算 loss。
- 当前 token embedding 与 LM-head weights 独立，不引入 weight tying。
- 本轮只准备 Step 15.4；不进入 Step 15.5，不做完整逐行 forward shape trace、整个 MiniGPT parameter count，也不逐行连接全部历史目的。
